In [47]:
import torch.nn as nn
import torch

In [48]:
class RotaryPositionalEmbeddings(nn.Module):

  def __init__(self, d: int, base: int = 10_000):

    super().__init__()
    self.base = base
    self.d = d
    self.cos_cached = None
    self.sin_cached = None

  def _build_cache(self, x: torch.Tensor):

    if self.cos_cached is not None and x.shape[0] <= self.cos_cached.shape[0]:
      return

    seq_len = x.shape[0]

    theta = 1.0 / (self.base ** (torch.arange(0, self.d, 2).float() / self.d)).to(x.device) # THETA = 10,000^(-2*i/d) = 1/10,000^(2i/d)

    seq_idx = torch.arange(seq_len, device=x.device).float().to(x.device) # Position Index -> [0,1,2...seq-1]

    idx_theta = torch.einsum('n,d->nd', seq_idx, theta)  # Calculates m*(THETA) = [ [0, 0...], [THETA_1, THETA_2...THETA_d/2], ... [seq-1*(THETA_1), seq-1*(THETA_2)...] ]
    print("\nidx_theta", idx_theta)

    idx_theta2 = torch.cat([idx_theta, idx_theta], dim=1) # [THETA_1, THETA_2...THETA_d/2] -> [THETA_1, THETA_2...THETA_d]
    print("\nidx_theta2", idx_theta2)

    self.cos_cached = idx_theta2.cos()[:, None, None, :] #Cache [cosTHETA_1, cosTHETA_2...cosTHETA_d]
    self.sin_cached = idx_theta2.sin()[:, None, None, :] #cache [sinTHETA_1, sinTHETA_2...sinTHETA_d]
    print("\ncos_cached", self.cos_cached)
    print("\nsin_cached", self.sin_cached)

  def _neg_half(self, x: torch.Tensor):

    d_2 = self.d // 2

    print("\nneg_half", torch.cat([-x[:, :, :, d_2:], x[:, :, :, :d_2]], dim=-1))
    return torch.cat([-x[:, :, :, d_2:], x[:, :, :, :d_2]], dim=-1) # [x_1, x_2,...x_d] -> [-x_d/2, ... -x_d, x_1, ... x_d/2]


  def forward(self, x: torch.Tensor):

    self._build_cache(x)

    neg_half_x = self._neg_half(x)

    x_rope = (x * self.cos_cached[:x.shape[0]]) + (neg_half_x * self.sin_cached[:x.shape[0]]) # type: ignore # [x_1*cosTHETA_1 - x_d/2*sinTHETA_d/2, ....]

    return x_rope

In [ ]:
# x = torch.tensor([[1, 2, 3, 4], [4, 5, 6, 7], [7, 8, 9, 10]], dtype=torch.float)
x = torch.randn((3))
print(x.shape)
x = x[:, None, None, :]
print(x.shape)

torch.Size([3, 8])
torch.Size([3, 1, 1, 8])


In [50]:
RotaryPositionalEmbeddings(8)(x)


idx_theta tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.0000e-01, 1.0000e-02, 1.0000e-03],
        [2.0000e+00, 2.0000e-01, 2.0000e-02, 2.0000e-03]])

idx_theta2 tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.0000e-01, 1.0000e-02, 1.0000e-03, 1.0000e+00, 1.0000e-01,
         1.0000e-02, 1.0000e-03],
        [2.0000e+00, 2.0000e-01, 2.0000e-02, 2.0000e-03, 2.0000e+00, 2.0000e-01,
         2.0000e-02, 2.0000e-03]])

cos_cached tensor([[[[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
            1.0000]]],


        [[[ 0.5403,  0.9950,  0.9999,  1.0000,  0.5403,  0.9950,  0.9999,
            1.0000]]],


        [[[-0.4161,  0.9801,  0.9998,  1.0000, -0.4161,  0.9801,  0.9998,
            1.0000]]]])

sin_cached tensor([[[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]]],


        [[[0.8415, 0.0998, 0.0100, 0.0010, 0.8415, 0.099

tensor([[[[-0.9270, -0.4062,  0.3291,  0.1854,  0.0907,  0.4967, -1.3021,
            0.5357]]],


        [[[ 0.0743, -0.5359,  0.4847,  0.1068, -0.6757,  0.1784,  0.5600,
            0.3055]]],


        [[[-0.3817, -0.0941,  0.5255, -2.2973, -1.3925,  0.2255, -0.0191,
            1.2677]]]])

In [51]:
print(torch.randn((2,4,4,1)))
print(torch.randn((2,4,4,1)).squeeze(3))

tensor([[[[ 0.1737],
          [ 0.2732],
          [ 0.4670],
          [-0.1423]],

         [[ 0.1598],
          [-0.2371],
          [-0.2101],
          [-0.9248]],

         [[ 0.1998],
          [ 1.0563],
          [ 0.5364],
          [-0.5307]],

         [[-0.2324],
          [ 1.1730],
          [ 1.1085],
          [ 1.2775]]],


        [[[-1.3446],
          [ 0.7536],
          [ 0.7617],
          [-0.0316]],

         [[ 0.1098],
          [ 0.3429],
          [ 1.2849],
          [-0.9703]],

         [[ 1.5175],
          [ 0.7803],
          [ 0.2129],
          [-0.6049]],

         [[-0.5914],
          [-0.4139],
          [ 0.5062],
          [-0.4364]]]])
tensor([[[ 0.2806, -1.3921, -0.4618,  1.2875],
         [-1.2743,  0.0439, -0.3485,  1.5259],
         [ 0.4008,  1.3251, -0.8047, -1.1257],
         [-0.6432, -1.5671,  0.6118, -1.5923]],

        [[ 2.0466,  0.0376, -1.2018,  0.3860],
         [-0.6382, -0.3619, -1.4402,  1.0796],
         [-0.0162,  0.821

In [52]:
RotaryPositionalEmbeddings(4)(torch.randn((2,4,4,1))).shape


idx_theta tensor([[0.0000, 0.0000],
        [1.0000, 0.0100]])

idx_theta2 tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0100, 1.0000, 0.0100]])

cos_cached tensor([[[[1.0000, 1.0000, 1.0000, 1.0000]]],


        [[[0.5403, 0.9999, 0.5403, 0.9999]]]])

sin_cached tensor([[[[0.0000, 0.0000, 0.0000, 0.0000]]],


        [[[0.8415, 0.0100, 0.8415, 0.0100]]]])

neg_half tensor([[[[-1.7040],
          [ 0.4476],
          [ 0.9414],
          [-0.9642]],

         [[ 0.8619],
          [-0.9219],
          [-1.4598],
          [-0.2062]],

         [[-0.0815],
          [ 0.4777],
          [-0.7245],
          [-0.8138]],

         [[-0.9731],
          [-0.7277],
          [-0.6930],
          [-0.6943]]],


        [[[-2.0107],
          [ 0.7938],
          [ 0.5216],
          [-0.5083]],

         [[ 1.1146],
          [-0.7971],
          [ 0.4046],
          [-1.0171]],

         [[-0.0857],
          [ 0.7424],
          [ 0.8715],
          [ 0.4314]],

         [[

torch.Size([2, 4, 4, 4])

In [53]:
import numpy as np

def get_emb(sin_inp):
    """
    Positional encoding code from tatp22/multidim-positional-encoding)

    Gets a base embedding for one dimension with sin and cos intertwined
    """
    emb = torch.stack((sin_inp.sin(), sin_inp.cos()), dim=-1)
    return torch.flatten(emb, -2, -1)

class PositionalEncoding1D(nn.Module):
    def __init__(self, channels, dtype_override=None):
        super(PositionalEncoding1D, self).__init__()
        self.org_channels = channels
        channels = int(np.ceil(channels / 2) * 2)
        self.channels = channels

        inv_freq = 1.0 / (10000 ** (torch.arange(0, channels, 2).float() / channels))

        self.register_buffer("inv_freq", inv_freq)
        self.register_buffer("cached_posenc", None, persistent=False)

        self.dtype_override = dtype_override

    def forward(self, tensor):
        if len(tensor.shape) != 3:
            raise RuntimeError("The input tensor has to be 3D!")
        if self.cached_posenc is not None and self.cached_posenc.shape == tensor.shape:
            return self.cached_posenc

        self.cached_posenc = None
        batch_size, x, orig_ch = tensor.shape
        pos_x = torch.arange(x, device=tensor.device, dtype=self.inv_freq.dtype)  # type: ignore
        sin_inp_x = torch.einsum("i,j->ij", pos_x, self.inv_freq)
        emb_x = get_emb(sin_inp_x)
        emb = torch.zeros(
            (x, self.channels),
            device=tensor.device,
            dtype=(
                self.dtype_override if self.dtype_override is not None else tensor.dtype
            ),
        )
        emb[:, : self.channels] = emb_x
        print(emb)

        self.cached_posenc = emb[None, :, :orig_ch].repeat(batch_size, 1, 1)
        return self.cached_posenc


class Summer(nn.Module):
    def __init__(self, penc):
        super(Summer, self).__init__()
        self.posenc = penc

    def forward(self, tensor):
        posenc = self.posenc(tensor)
        assert (
            tensor.size() == posenc.size()
        ), f"The original tensor size {tensor.size()} and the positional encoding tensor size {posenc.size()} must match!"
        return tensor + posenc.to(tensor.device)

In [54]:
PositionalEncoding1D(channels=4)(torch.randn(2,3,4))

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998]])


tensor([[[ 0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.5403,  0.0100,  0.9999],
         [ 0.9093, -0.4161,  0.0200,  0.9998]],

        [[ 0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.5403,  0.0100,  0.9999],
         [ 0.9093, -0.4161,  0.0200,  0.9998]]])

In [ ]:
import torch
import torch.nn as nn

class RotaryPositionalEmbeddings1D(nn.Module):
    def __init__(self, channels: int, base: int = 10_000):
        super().__init__()
        self.channels = channels
        self.base = base
        
        self.register_buffer("cos_cached", None, persistent=False)
        self.register_buffer("sin_cached", None, persistent=False)

    def _build_cache(self, x: torch.Tensor):
        # Input x is [B, T, C], so sequence length (Time) is dimension 1
        seq_len = x.shape[1]

        if self.cos_cached is not None and seq_len <= self.cos_cached.shape[1]: # type: ignore
            return

        # THETA = 1/10,000^(2i/d)
        theta = 1.0 / (self.base ** (torch.arange(0, self.channels, 2).float() / self.channels)).to(x.device)

        # Position Index -> [0, 1, 2 ... T-1]
        seq_idx = torch.arange(seq_len, device=x.device).float()

        # Calculates [T, channels/2]
        idx_theta = torch.einsum('n,d->nd', seq_idx, theta)

        # Concatenate to match channel dimension: [T, channels/2] -> [T, channels]
        idx_theta2 = torch.cat([idx_theta, idx_theta], dim=1)

        cos_cache = idx_theta2.cos().unsqueeze(0) # [1, T, C]
        sin_cache = idx_theta2.sin().unsqueeze(0) # [1, T, C]

        self.register_buffer("cos_cached", cos_cache, persistent=False)
        self.register_buffer("sin_cached", sin_cache, persistent=False)

    def _neg_half(self, x: torch.Tensor):
        d_2 = self.channels // 2
        
        # Using the ellipsis `...` allows this to work dynamically regardless 
        # of whether the input is 2D, 3D [B, T, C], or 4D
        return torch.cat([-x[..., d_2:], x[..., :d_2]], dim=-1)

    def forward(self, x: torch.Tensor):
        if len(x.shape) != 3:
            raise RuntimeError("The input tensor has to be 3D [Batch, Time, Channels]!")
            
        self._build_cache(x)
        
        seq_len = x.shape[1]
        
        # cos and sin will be of shape [1, T, C]
        cos = self.cos_cached[:, :seq_len, :]  # type: ignore
        sin = self.sin_cached[:, :seq_len, :]  # type: ignore

        neg_half_x = self._neg_half(x)

        x_rope = (x * cos) + (neg_half_x * sin)

        return x_rope